# 01 · Two-stage anchor-based: Faster R-CNN (ResNet-50-FPN v2, torchvision)
Fixed protocol shared by all detectors: train on every 5th frame of the train clips (ignored regions masked),
evaluate on every 5th frame of the 7 val clips with `common.CocoEvaluator`.

Kaggle: Accelerator **GPU**, **Internet on**. Run with *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
sys.path.insert(0, next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists()))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from common import *

seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2') if not FAST_DEV else Path('work_dev')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '', '| torch', torch.__version__, '| torchvision', torchvision.__version__)

## 1. Protocol (same for all detectors)

In [ ]:
STEP = 5             # frame step; justified in 00_eda (consecutive frames nearly identical)
EPOCHS = 4 if not FAST_DEV else 1
BATCH = 4 if not FAST_DEV else 2
LR = 0.01            # SGD, warm-up + cosine, AMP
PRETRAINED = not FAST_DEV

yaml_path = prepare_training_copy(ROOT, TMP / 'data', step_train=STEP, step_val=STEP)
manifest = TMP / 'data' / 'manifest.csv'
train_ds = build_torch_dataset(manifest, 'train', train=True)
val_frames = list_frames(ROOT / 'val', STEP)          # raw frames for evaluation
evaluator = CocoEvaluator(ROOT, val_frames)
print('train images:', len(train_ds), '| val frames:', len(val_frames))

## 2. Model builder and experiments
* `base` – torchvision defaults (anchors 32–512 px, 2000/1000 RPN proposals train/test, 512 RoIs, 25% positives).
* `rpn_more` – more RPN proposals (dense scenes: up to hundreds of objects per frame).
* `small_anchors` – anchors halved (16–256 px) because most objects are < 32 px.
* `roi_pos50` – RoI sampling: 256 RoIs with 50% positives (more positive RoIs per image).

Detections per image = 500 and score threshold 0.001 for **all** runs (leaderboard scores up to 500 boxes).

In [ ]:
EXPERIMENTS = {
    'frcnn_base':          dict(),
    'frcnn_rpn_more':      dict(rpn_pre_nms_top_n_train=4000, rpn_post_nms_top_n_train=4000,
                                rpn_pre_nms_top_n_test=3000, rpn_post_nms_top_n_test=2000),
    'frcnn_small_anchors': dict(anchor_sizes=((16,), (32,), (64,), (128,), (256,))),
    'frcnn_roi_pos50':     dict(box_batch_size_per_image=256, box_positive_fraction=0.5),
}
RUN = ['frcnn_base', 'frcnn_rpn_more', 'frcnn_small_anchors']  # add 'frcnn_roi_pos50' if GPU time allows
if FAST_DEV:
    RUN = RUN[:2]


def build_frcnn(cfg):
    cfg = dict(cfg)
    anchor_sizes = cfg.pop('anchor_sizes', None)
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1 if PRETRAINED else None,
                                       weights_backbone=None, box_detections_per_img=500, box_score_thresh=0.001,
                                       **(dict(min_size=320, max_size=512) if FAST_DEV else {}), **cfg)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NC + 1)  # 8 classes + background
    if anchor_sizes:  # same number of anchors per location (3) -> pretrained RPN head is reused
        model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, ((0.5, 1.0, 2.0),) * len(anchor_sizes))
    return model

## 3. Train and evaluate

In [ ]:
histories, results = {}, {}
for name in RUN:
    print(f'\n===== {name} =====')
    seed_everything()
    model = build_frcnn(EXPERIMENTS[name])
    histories[name] = train_torchvision(model, train_ds, evaluator, val_frames, DEVICE, epochs=EPOCHS, lr=LR,
                                        batch=BATCH, out_dir=TMP / 'runs' / name)
    preds, fps = predict_torchvision(model, val_frames, DEVICE)
    m = evaluator.evaluate(preds)
    results[name] = preds
    log_result(dict(model=name, family='two-stage', epochs=EPOCHS, step=STEP, fps=fps, **m), OUT / 'metrics.csv')
    pd.DataFrame(histories[name]).to_csv(OUT / f'history_{name}.csv', index=False)
    pickle.dump(preds, open(TMP / f'val_preds_{name}.pkl', 'wb'))
    print(name, {k: m[k] for k in ['mAP', 'AP50', 'AP_small']}, 'fps', fps)
    del model; torch.cuda.empty_cache()

## 4. Results

In [ ]:
tab = pd.read_csv(OUT / 'metrics.csv')
cols = ['model', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AR500', 'fps']
print(tab[tab.family == 'two-stage'][cols].to_string(index=False))
print('\nper-class AP:')
print(tab[tab.family == 'two-stage'][['model'] + [f'AP_{c}' for c in CLASSES]].to_string(index=False))

plt.figure(figsize=(6, 3.5))
for name, h in histories.items():
    h = pd.DataFrame(h); plt.plot(h.epoch, h.mAP, marker='o', label=name)
plt.xlabel('epoch'); plt.ylabel('val mAP'); plt.legend(); plt.title('Faster R-CNN: val mAP per epoch')
plt.tight_layout(); plt.savefig(FIG / 'frcnn_curves.png', dpi=120); plt.show()

BEST = tab[tab.family == 'two-stage'].sort_values('mAP').model.iloc[-1]
print('best two-stage run:', BEST)
print(evaluator.evaluate_by_size(results[BEST]).to_string(index=False))

## 5. What each stage contributes
RPN = class-agnostic proposals; the second stage classifies them and refines boxes.
We measure the recall of the RPN proposals alone (top-N, IoU ≥ 0.5) and of the final detections.

In [ ]:
model = build_frcnn(EXPERIMENTS[BEST]); model.load_state_dict(torch.load(TMP / 'runs' / BEST / 'best.pt', map_location=DEVICE))
model.eval().to(DEVICE)
sub = val_frames[::max(1, len(val_frames) // 60)]
rec = {n: [] for n in [100, 300, 1000, 'final@0.3']}
with torch.no_grad():
    for c, p in sub:
        img = cv2.imread(str(p))[:, :, ::-1]; h, w = img.shape[:2]
        gt, _ = read_yolo_labels(label_path_for(ROOT, 'val', c, p), w, h)
        if len(gt) == 0:
            continue
        x = torch.from_numpy(img.copy()).permute(2, 0, 1).float().div(255).to(DEVICE)
        images, _ = model.transform([x])
        feats = model.backbone(images.tensors)
        props, _ = model.rpn(images, feats)
        sy, sx = h / images.image_sizes[0][0], w / images.image_sizes[0][1]
        props = props[0].cpu().numpy() * np.array([sx, sy, sx, sy])
        for n in [100, 300, 1000]:
            rec[n].append((box_iou_np(gt, props[:n]).max(1) >= 0.5).mean() if len(props) else 0.0)
        b, s, _ = results[BEST][frame_key(c, p)]
        rec['final@0.3'].append((box_iou_np(gt, b[s >= 0.3]).max(1) >= 0.5).mean() if (s >= 0.3).any() else 0.0)
print({k: round(float(np.mean(v)), 3) for k, v in rec.items()})
del model; torch.cuda.empty_cache()

## 6. Failure cases
Per frame: missed GT, false positives (outside ignored regions) and wrong class at score ≥ 0.3.
Below: the frames with the most misses, the most false positives, and a class-confusion table.

In [ ]:
err = frame_errors(ROOT, val_frames, results[BEST], score_thr=0.3)
print(err[['missed', 'fp', 'wrong_cls']].sum())
worst = [err.sort_values('missed').iloc[-1], err.sort_values('fp').iloc[-1], err.sort_values('wrong_cls').iloc[-1]]
fig, axes = plt.subplots(3, 1, figsize=(14, 24))
for ax, r, why in zip(axes, worst, ['most missed', 'most false positives', 'most wrong class']):
    show_frame(ROOT, r['path'], r['clip'], results[BEST], 0.3, ax=ax, title=f'{why}: {r.key}  (GT {r.n_gt}, missed {r.missed}, FP {r.fp}, wrong cls {r.wrong_cls})')
plt.tight_layout(); plt.savefig(FIG / 'frcnn_failures.png', dpi=90); plt.show()

In [ ]:
# class confusion of matched pairs (IoU >= 0.5, score >= 0.3); rows = GT, cols = prediction
conf = np.zeros((NC, NC), int)
for c, p in val_frames:
    img_h, img_w = cv2.imread(str(p)).shape[:2]
    gt, gc = read_yolo_labels(label_path_for(ROOT, 'val', c, p), img_w, img_h)
    b, s, l = results[BEST][frame_key(c, p)]
    b, l = b[s >= 0.3], l[s >= 0.3]
    if len(gt) and len(b):
        iou = box_iou_np(gt, b)
        ok = iou.max(1) >= 0.5
        np.add.at(conf, (gc[ok], l[iou.argmax(1)[ok]]), 1)
print(pd.DataFrame(conf, index=CLASSES, columns=CLASSES))

## 7. Notes for the report
*(your own analysis: what RPN vs second stage contribute (section 5), effect of each tuned parameter,
three failure cases with plausible causes)*